# 01 — Data Exploration

Inspect the five paired prompt datasets before running experiments.

Goals:
- Verify prompt pair structure and manipulation quality
- Check dataset balance across domains and difficulty levels
- Inspect a sample of each condition to confirm biases are injected correctly
- Compute baseline model accuracy on control prompts (is the model actually correct before manipulation?)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from src.data.loaders import load_all_pairs
from src.data.prompt_templates import DatasetType

In [ ]:
# Build datasets if not already built
from src.data.dataset_builder import DatasetBuilder
builder = DatasetBuilder(output_dir='../data/processed', min_pairs=500, seed=42)
paths = builder.build_all()
for dt, p in paths.items():
    print(f'{dt.value}: {p}')

In [ ]:
# Load all datasets
all_pairs = load_all_pairs(data_dir='../data/processed')
for dt, pairs in all_pairs.items():
    print(f'{dt.value}: {len(pairs)} pairs')

In [ ]:
# Inspect 3 samples from each dataset
for dt, pairs in all_pairs.items():
    print(f'\n{"="*60}')
    print(f'Dataset: {dt.value}')
    for pair in random.sample(pairs, min(3, len(pairs))):
        print(f'\n  ID: {pair.prompt_id}')
        print(f'  CONTROL (first 200 chars): {pair.control[:200]}')
        print(f'  TREATMENT (first 200 chars): {pair.treatment[:200]}')
        print(f'  CORRECT ANSWER: {pair.correct_answer}')

In [ ]:
# Check control/treatment length distributions
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (dt, pairs) in zip(axes, all_pairs.items()):
    ctrl_lens = [len(p.control) for p in pairs]
    trt_lens = [len(p.treatment) for p in pairs]
    ax.hist(ctrl_lens, alpha=0.5, label='control', bins=30)
    ax.hist(trt_lens, alpha=0.5, label='treatment', bins=30)
    ax.set_title(dt.value, fontsize=9)
    ax.set_xlabel('prompt length (chars)')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig('../paper/figures/data_length_distributions.png', dpi=150)
plt.show()

In [ ]:
# Baseline model accuracy on control prompts
# This validates that the model is actually correct before manipulation.
# Run only on a sample to avoid long execution time in this notebook.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_HF_ID = 'google/gemma-3-4b-it'
SAMPLE_N = 50
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_HF_ID, torch_dtype=torch.bfloat16, device_map=device
)
model.eval()

def generate_answers(prompts, max_new_tokens=256):
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return [
        tokenizer.decode(o[inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        for o in out
    ]

from src.interventions.evaluation import _answers_match

for dt, pairs in all_pairs.items():
    sample = pairs[:SAMPLE_N]
    ctrl_prompts = [p.control for p in sample]
    trt_prompts = [p.treatment for p in sample]
    correct = [p.correct_answer for p in sample]
    
    ctrl_outputs = generate_answers(ctrl_prompts)
    trt_outputs = generate_answers(trt_prompts)
    
    ctrl_acc = sum(_answers_match(o, a) for o, a in zip(ctrl_outputs, correct)) / len(correct)
    trt_acc = sum(_answers_match(o, a) for o, a in zip(trt_outputs, correct)) / len(correct)
    
    print(f'{dt.value}: control accuracy={ctrl_acc:.2f}, treatment accuracy={trt_acc:.2f}, delta={trt_acc-ctrl_acc:+.2f}')